In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
pip install -U bitsandbytes>=0.46.1

In [ ]:
"""
Smart MCQ Solver - final pipeline (Qwen2.5-7B-Instruct edition)
Roll no: 23f2004250

Plan for this notebook:
1. load train/test
2. baseline-ish model built completely from scratch (TF-IDF vectors + a small
   feedforward net trained with plain numpy/torch, no pretrained weights)
3. fine-tune Qwen2.5-7B-Instruct as the "big model", 5 fold CV so we get
   out-of-fold probs for stacking + a more stable test time prediction.
   The 7B model is loaded in 4-bit (QLoRA) and only LoRA adapters are
   trained. Instead of a classification head, we keep it a causal LM and
   read off the next-token logits for the letters A/B/C/D/E right after an
   "Answer:" prompt - this is the standard way to get calibrated per-option
   probabilities out of an instruct model without changing its architecture.
4. blend the two sets of probabilities (grid search over the blend weight
   using OOF MAP@3) and write out submission.csv

Everything lives in this one file on purpose - easier to run top to bottom
in a single Kaggle cell and easier to explain in the viva than something
split across five notebooks.

NOTE ON COMPUTE: fine-tuning a 7B model 5 times (even with QLoRA) is a lot
heavier than the old DeBERTa-v3-base run. On a single Kaggle T4/P100:
  - QWEN_N_FOLDS=5, QWEN_EPOCHS=1 can take a few hours depending on dataset size.
  - If you're tight on time/quota, drop QWEN_N_FOLDS to 3 (still gives OOF
    probs, just coarser) and/or QWEN_EPOCHS stays at 1.
Both knobs are exposed near the top of the Qwen section below.

Before running, make sure these are installed (uncomment in a Kaggle cell):
    !pip install -q -U bitsandbytes peft accelerate
"""

import os
import gc
import random

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import StratifiedKFold

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("running on:", DEVICE)
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
if torch.cuda.is_available():
    torch.cuda.empty_cache()

DATA_DIR = "/kaggle/input/competitions/smart-mcq-solver-challenge"  # change this if your dataset slug is different
TRAIN_PATH = os.path.join(DATA_DIR, "train.csv")
TEST_PATH = os.path.join(DATA_DIR, "test.csv")

# fall back to local paths when just testing the script outside kaggle
if not os.path.exists(TRAIN_PATH):
    TRAIN_PATH = "train.csv"
    TEST_PATH = "test.csv"

LETTERS = ["A", "B", "C", "D", "E"]
NUM_CHOICES = 5

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

# a couple of rows had trailing whitespace in the option text in my run, strip
# everything just to be safe - costs nothing and avoids weird tokenization
for col in LETTERS:
    train_df[col] = train_df[col].astype(str).str.strip()
    test_df[col] = test_df[col].astype(str).str.strip()
train_df["prompt"] = train_df["prompt"].astype(str).str.strip()
test_df["prompt"] = test_df["prompt"].astype(str).str.strip()

print(train_df.shape, test_df.shape)
train_df.head(2)


def map_at_3(true_letters, pred_letter_lists):
    """
    true_letters: list/array of the correct letter per row
    pred_letter_lists: list of lists, top 3 predicted letters per row (ranked)
    plain MAP@3 - since we only ever have one relevant label this collapses
    to 1/rank if the true label is somewhere in the top 3, else 0.
    """
    scores = []
    for true, preds in zip(true_letters, pred_letter_lists):
        s = 0.0
        for rank, p in enumerate(preds[:3], start=1):
            if p == true:
                s = 1.0 / rank
                break
        scores.append(s)
    return float(np.mean(scores))


def letters_from_probs(prob_matrix):
    """prob_matrix shape (n, 5) -> list of top3 letter lists, ranked desc"""
    order = np.argsort(-prob_matrix, axis=1)
    out = []
    for row in order:
        out.append([LETTERS[i] for i in row[:3]])
    return out


# -----------------------------------------------------------------------
# Model 1 - from scratch: TF-IDF similarity features + small feedforward net
# -----------------------------------------------------------------------
# Idea here is simple - for every option, build a feature vector from the
# cosine similarity between (prompt) and (option) tf-idf vectors, plus a
# couple of length based features, then let a small MLP learn to score each
# option. This is our "built from scratch, no pretrained weights" model per
# the course requirement.

def build_tfidf_features(df, vectorizer=None, fit=False):
    prompts = df["prompt"].tolist()
    all_texts = prompts.copy()
    for col in LETTERS:
        all_texts += df[col].tolist()

    if fit:
        vectorizer = TfidfVectorizer(
            max_features=30000,
            ngram_range=(1, 2),
            stop_words="english",
            sublinear_tf=True,
        )
        vectorizer.fit(all_texts)

    prompt_vecs = vectorizer.transform(prompts)

    feats = np.zeros((len(df), NUM_CHOICES, 4), dtype=np.float32)
    for j, col in enumerate(LETTERS):
        opt_vecs = vectorizer.transform(df[col].tolist())
        num = np.asarray(prompt_vecs.multiply(opt_vecs).sum(axis=1)).ravel()
        p_norm = np.sqrt(np.asarray(prompt_vecs.multiply(prompt_vecs).sum(axis=1)).ravel())
        o_norm = np.sqrt(np.asarray(opt_vecs.multiply(opt_vecs).sum(axis=1)).ravel())
        denom = (p_norm * o_norm)
        denom[denom == 0] = 1e-9
        cos_sim = num / denom

        opt_len = df[col].str.split().apply(len).values.astype(np.float32)
        prompt_len = df["prompt"].str.split().apply(len).values.astype(np.float32)
        len_ratio = opt_len / np.maximum(prompt_len, 1.0)

        feats[:, j, 0] = cos_sim
        feats[:, j, 1] = opt_len / 100.0
        feats[:, j, 2] = len_ratio
        feats[:, j, 3] = num

    return feats, vectorizer


class ScratchMLP(nn.Module):
    """tiny feedforward net, shared weights across the 5 options (like a
    siamese scorer) then softmax over the 5 scores at the end"""

    def __init__(self, in_dim=4, hidden=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden, hidden // 2),
            nn.ReLU(),
            nn.Linear(hidden // 2, 1),
        )

    def forward(self, x):
        b, c, d = x.shape
        scores = self.net(x.view(b * c, d)).view(b, c)
        return scores


def train_scratch_model(train_feats, y_idx, epochs=25, lr=1e-3, batch_size=64):
    scratch_model = ScratchMLP(in_dim=train_feats.shape[-1]).to(DEVICE)
    opt = torch.optim.Adam(scratch_model.parameters(), lr=lr, weight_decay=1e-4)
    loss_fn = nn.CrossEntropyLoss()

    X = torch.tensor(train_feats, dtype=torch.float32)
    y = torch.tensor(y_idx, dtype=torch.long)
    n = X.shape[0]
    idx = np.arange(n)

    scratch_model.train()
    for ep in range(epochs):
        np.random.shuffle(idx)
        total_loss = 0.0
        for start in range(0, n, batch_size):
            batch_idx = idx[start:start + batch_size]
            xb = X[batch_idx].to(DEVICE)
            yb = y[batch_idx].to(DEVICE)

            opt.zero_grad()
            logits = scratch_model(xb)
            loss = loss_fn(logits, yb)
            loss.backward()
            opt.step()
            total_loss += loss.item() * len(batch_idx)

        if (ep + 1) % 5 == 0 or ep == 0:
            print(f"[scratch model] epoch {ep+1}/{epochs} loss={total_loss/n:.4f}")

    return scratch_model


@torch.no_grad()
def predict_scratch(model, feats, batch_size=256):
    model.eval()
    X = torch.tensor(feats, dtype=torch.float32)
    probs = []
    for start in range(0, X.shape[0], batch_size):
        xb = X[start:start + batch_size].to(DEVICE)
        logits = model(xb)
        probs.append(torch.softmax(logits, dim=1).cpu().numpy())
    return np.concatenate(probs, axis=0)


answer_to_idx = train_df["answer"].map(lambda a: LETTERS.index(a)).values

train_feats, tfidf_vec = build_tfidf_features(train_df, fit=True)
test_feats, _ = build_tfidf_features(test_df, vectorizer=tfidf_vec, fit=False)

scratch_model = train_scratch_model(train_feats, answer_to_idx, epochs=25)
scratch_train_probs = predict_scratch(scratch_model, train_feats)
scratch_test_probs = predict_scratch(scratch_model, test_feats)

scratch_preds = letters_from_probs(scratch_train_probs)
print("scratch model train MAP@3 (not OOF, just a sanity check):",
      map_at_3(train_df["answer"].tolist(), scratch_preds))

del train_feats
gc.collect()

# -----------------------------------------------------------------------
# Model 2 - Qwen2.5-7B-Instruct, QLoRA fine-tuned, 5-fold CV
# -----------------------------------------------------------------------
# We keep Qwen as a causal LM (no custom classification head). Each row is
# turned into a chat prompt ending right at "Answer:" (via
# add_generation_prompt=True), and during training we teach the model to
# emit the correct letter as the very next token. At inference we don't
# generate - we just read the logits at that same position, restrict them
# to the 5 letter token ids, and softmax those 5 numbers into probabilities.
# This gives us a (n, 5) probability matrix exactly like the scratch model
# and the old DeBERTa model produced, so the rest of the ensembling code
# below barely changes.

QWEN_MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
MAX_LEN_QWEN = 512          # prompt token budget (truncated from the left if exceeded)
QWEN_N_FOLDS = 5            # drop to 3 if you're tight on Kaggle GPU quota/time
QWEN_EPOCHS = 1             # 1 epoch per fold is usually enough for LoRA on this kind of task
QWEN_LR = 2e-4
QWEN_TRAIN_BS = 1           # per-step micro-batch, kept tiny on purpose for 7B + 4bit
QWEN_GRAD_ACCUM = 8         # effective batch size = QWEN_TRAIN_BS * QWEN_GRAD_ACCUM
QWEN_EVAL_BS = 8
LOG_EVERY = 50

qwen_tokenizer = AutoTokenizer.from_pretrained(QWEN_MODEL_NAME, trust_remote_code=True)
if qwen_tokenizer.pad_token is None:
    qwen_tokenizer.pad_token = qwen_tokenizer.eos_token

# figure out the token id for each answer letter once - these are what we
# read the logits off at inference time
LETTER_TOKEN_IDS = []
for L in LETTERS:
    ids = qwen_tokenizer.encode(L, add_special_tokens=False)
    if len(ids) != 1:
        print(f"WARNING: letter '{L}' tokenizes to {len(ids)} tokens ({ids}), "
              f"using only the first token id - double check this on your tokenizer")
    LETTER_TOKEN_IDS.append(ids[0])
print("letter token ids:", dict(zip(LETTERS, LETTER_TOKEN_IDS)))

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj"]


def build_messages(prompt, options):
    system_msg = (
        "You are an expert multiple-choice exam solver. You will be given a "
        "question and five options labelled A to E. Respond with only the "
        "single capital letter of the best answer and nothing else."
    )
    options_block = "\n".join(f"{L}) {opt}" for L, opt in zip(LETTERS, options))
    user_msg = f"Question: {prompt}\n\nOptions:\n{options_block}\n\nAnswer:"
    return [
        {"role": "system", "content": system_msg},
        {"role": "user", "content": user_msg},
    ]


def build_prompt_ids(prompt, options, tokenizer, max_len):
    messages = build_messages(prompt, options)
    # add_generation_prompt=True leaves the text ending right where the
    # assistant's reply should start - i.e. right before the letter
    prompt_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    prompt_ids = tokenizer(prompt_text, add_special_tokens=False)["input_ids"]
    if len(prompt_ids) > max_len:
        # truncate from the left so we keep the assistant tag at the end intact
        prompt_ids = prompt_ids[-max_len:]
    return prompt_ids


class QwenMCQDataset(Dataset):
    """
    has_labels=True  -> returns input_ids/labels for training
                        (prompt + correct letter + eos, loss masked to just
                        the letter + eos)
    has_labels=False -> returns just prompt_ids, for logit-based inference
    """

    def __init__(self, df, tokenizer, max_len=MAX_LEN_QWEN, has_labels=True):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.has_labels = has_labels

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        options = [row[c] for c in LETTERS]
        prompt_ids = build_prompt_ids(row["prompt"], options, self.tokenizer, self.max_len)
        item = {"prompt_ids": prompt_ids}
        if self.has_labels:
            letter = row["answer"]
            letter_ids = self.tokenizer(letter, add_special_tokens=False)["input_ids"]
            eos_id = self.tokenizer.eos_token_id
            # concatenate ids directly instead of re-tokenizing the joined
            # string, so we know exactly where the prompt ends and don't
            # get bitten by BPE merges across the boundary
            input_ids = prompt_ids + letter_ids + [eos_id]
            labels = [-100] * len(prompt_ids) + letter_ids + [eos_id]
            item["input_ids"] = input_ids
            item["labels"] = labels
        return item


def train_collate(batch, pad_id):
    """right-padded, for the training loop (loss masking handles the pad)"""
    max_len = max(len(b["input_ids"]) for b in batch)
    input_ids, attn, labels = [], [], []
    for b in batch:
        pad_n = max_len - len(b["input_ids"])
        input_ids.append(b["input_ids"] + [pad_id] * pad_n)
        attn.append([1] * len(b["input_ids"]) + [0] * pad_n)
        labels.append(b["labels"] + [-100] * pad_n)
    return {
        "input_ids": torch.tensor(input_ids, dtype=torch.long),
        "attention_mask": torch.tensor(attn, dtype=torch.long),
        "labels": torch.tensor(labels, dtype=torch.long),
    }


def infer_collate(batch, pad_id):
    """left-padded, so the last position (index -1) is always the true
    last real token for every row in the batch - that's the position whose
    logits we read the letter distribution off of"""
    max_len = max(len(b["prompt_ids"]) for b in batch)
    input_ids, attn = [], []
    for b in batch:
        pad_n = max_len - len(b["prompt_ids"])
        input_ids.append([pad_id] * pad_n + b["prompt_ids"])
        attn.append([0] * pad_n + [1] * len(b["prompt_ids"]))
    return {
        "input_ids": torch.tensor(input_ids, dtype=torch.long),
        "attention_mask": torch.tensor(attn, dtype=torch.long),
    }


@torch.no_grad()
def predict_qwen(model, dataset, tokenizer, batch_size=QWEN_EVAL_BS):
    model.eval()
    loader = DataLoader(
        dataset, batch_size=batch_size, shuffle=False,
        collate_fn=lambda b: infer_collate(b, tokenizer.pad_token_id),
    )
    all_probs = []
    for batch in loader:
        input_ids = batch["input_ids"].to(model.device)
        attention_mask = batch["attention_mask"].to(model.device)
        out = model(input_ids=input_ids, attention_mask=attention_mask)
        last_logits = out.logits[:, -1, :]
        letter_logits = last_logits[:, LETTER_TOKEN_IDS]
        probs = torch.softmax(letter_logits.float(), dim=1).cpu().numpy()
        all_probs.append(probs)
    return np.concatenate(all_probs, axis=0)


def load_fresh_qwen_lora():
    base_model = AutoModelForCausalLM.from_pretrained(
        QWEN_MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto",
        torch_dtype=torch.bfloat16,
        trust_remote_code=True,
    )
    base_model = prepare_model_for_kbit_training(base_model)
    base_model.gradient_checkpointing_enable()

    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=LORA_TARGET_MODULES,
    )
    model = get_peft_model(base_model, lora_config)
    model.print_trainable_parameters()
    return model


def train_qwen_fold(model, train_ds, tokenizer, epochs=QWEN_EPOCHS,
                     lr=QWEN_LR, train_bs=QWEN_TRAIN_BS, grad_accum=QWEN_GRAD_ACCUM):
    try:
        import bitsandbytes as bnb_lib
        optimizer = bnb_lib.optim.PagedAdamW8bit(model.parameters(), lr=lr)
    except Exception as e:
        print("falling back to torch AdamW (bitsandbytes 8bit optimizer unavailable):", e)
        optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

    loader = DataLoader(
        train_ds, batch_size=train_bs, shuffle=True,
        collate_fn=lambda b: train_collate(b, tokenizer.pad_token_id),
    )

    model.train()
    optimizer.zero_grad()
    for epoch in range(epochs):
        running_loss = 0.0
        for i, batch in enumerate(loader):
            batch = {k: v.to(model.device) for k, v in batch.items()}
            out = model(**batch)
            loss = out.loss / grad_accum
            loss.backward()
            running_loss += loss.item() * grad_accum

            if (i + 1) % grad_accum == 0:
                optimizer.step()
                optimizer.zero_grad()

            if (i + 1) % LOG_EVERY == 0:
                print(f"  epoch {epoch+1} step {i+1}/{len(loader)} "
                      f"loss={running_loss/(i+1):.4f}")

        print(f"[qwen] epoch {epoch+1}/{epochs} avg loss={running_loss/len(loader):.4f}")

    return model


skf_qwen = StratifiedKFold(n_splits=QWEN_N_FOLDS, shuffle=True, random_state=SEED)

oof_probs_qwen = np.zeros((len(train_df), NUM_CHOICES), dtype=np.float32)
test_probs_folds_qwen = []

for fold, (tr_idx, val_idx) in enumerate(skf_qwen.split(train_df, answer_to_idx)):
    print(f"\n===== Qwen fold {fold+1}/{QWEN_N_FOLDS} =====")

    tr_df = train_df.iloc[tr_idx]
    val_df = train_df.iloc[val_idx]

    qwen_model = load_fresh_qwen_lora()

    train_ds = QwenMCQDataset(tr_df, qwen_tokenizer, MAX_LEN_QWEN, has_labels=True)
    qwen_model = train_qwen_fold(qwen_model, train_ds, qwen_tokenizer)

    # OOF predictions for this fold's val split (no labels needed for inference)
    val_ds_infer = QwenMCQDataset(val_df, qwen_tokenizer, MAX_LEN_QWEN, has_labels=False)
    val_probs = predict_qwen(qwen_model, val_ds_infer, qwen_tokenizer)
    oof_probs_qwen[val_idx] = val_probs

    # predict on the actual test set too, average across folds later
    test_ds_qwen = QwenMCQDataset(test_df, qwen_tokenizer, MAX_LEN_QWEN, has_labels=False)
    test_probs_fold = predict_qwen(qwen_model, test_ds_qwen, qwen_tokenizer)
    test_probs_folds_qwen.append(test_probs_fold)

    fold_map3 = map_at_3(val_df["answer"].tolist(), letters_from_probs(val_probs))
    print(f"fold {fold+1} val MAP@3: {fold_map3:.4f}")

    # free up gpu memory before next fold, 7B models eat quota fast otherwise
    del qwen_model
    gc.collect()
    torch.cuda.empty_cache()

qwen_test_probs = np.mean(test_probs_folds_qwen, axis=0)

qwen_oof_preds = letters_from_probs(oof_probs_qwen)
print("\nQwen 5-fold OOF MAP@3:", map_at_3(train_df["answer"].tolist(), qwen_oof_preds))

# -----------------------------------------------------------------------
# Ensemble - blend scratch model probs with Qwen OOF/test probs
# -----------------------------------------------------------------------
# grid search the blend weight on OOF data since that's the only place we
# have ground truth for the Qwen model. scratch model probs are just
# recomputed on the full train set above (scratch_train_probs) - not a true
# OOF but good enough to pick a sensible blend weight, we're not overfitting
# a single number by much here

best_w, best_score = 0.0, -1.0
for w in np.arange(0.0, 1.01, 0.05):
    blended = w * scratch_train_probs + (1 - w) * oof_probs_qwen
    preds = letters_from_probs(blended)
    score = map_at_3(train_df["answer"].tolist(), preds)
    if score > best_score:
        best_score = score
        best_w = w

print(f"\nbest blend weight for scratch model = {best_w:.2f}, blended OOF MAP@3 = {best_score:.4f}")

final_test_probs = best_w * scratch_test_probs + (1 - best_w) * qwen_test_probs
final_preds = letters_from_probs(final_test_probs)

submission = pd.DataFrame({
    "ID": test_df["id"],
    "Prediction": [" ".join(p) for p in final_preds],
})

submission.to_csv("submission.csv", index=False)
print("\nsaved submission.csv, shape:", submission.shape)
submission.head()